# Adaptive Boosting - AdaBoost

In [4]:
import scipy.io as sio
import numpy as np
import plotly.graph_objects as go

# Load data for training
data = sio.loadmat("StairData.mat")
Xtr = data['data'][0][0][2]
ytr = data['data'][0][0][3].squeeze()

ytr=ytr.astype(int) # Labels

#Identify class indexes
ixW1 = np.where(ytr==1)[0]
ixW2 = np.where(ytr==2)[0]

ytr[ixW2] = -1 # Class 2 is labeled as -1

#Plot samples in the feature space
fig=go.Figure()
fig.add_trace(go.Scatter(x=Xtr[0,ixW1],y=Xtr[1,ixW1],name='1'))
fig.add_trace(go.Scatter(x=Xtr[0,ixW2],y=Xtr[1,ixW2],name='2'))
fig.update_traces(marker=dict(size=8))
fig.update_traces(mode='markers')
fig.update_xaxes(title_text="f1")
fig.update_yaxes(title_text="f2")
fig.update_layout(autosize=False,width=900,height=800)
fig.show()

In [5]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt
from sklearn import tree

def trainAdaboost(Xtr,ytr,nMod): # For training, we need the features, the labels and the number of models to train
    '''
    The function trains an Adaboost model using decision trees as weak learners.
    Xtr: training features (d x N)
    ytr: training labels (N)
    nMod: number of models to train
    return: array of trained models
    '''
    models = np.arange(nMod, dtype=DecisionTreeClassifier) # Initialize an array to store the trained models
    N = Xtr.shape[0] # Number of samples in the training data
    w = np.ones((N))*(1/N) # Initialize the weights of the samples to be equal
    alphas = np.zeros((nMod)) # Initialize the alpha values of the models to be zero
    
    for i in range(nMod):
        # Train a decision tree classifier with the current weights of the samples
        mdl = DecisionTreeClassifier(criterion='gini',max_depth=1)
        mdl.fit(Xtr,ytr,sample_weight=w)
        pred = mdl.predict(Xtr) # Predictions of the current model on the training data
        
        models[i] = mdl # Store the model
        
        err = sum(w*np.heaviside(-ytr*pred,1)) # Error is the sum of the weights of the misclassified samples
                                                # np.heaviside(-ytr*pred,1) is 1 for misclassified samples and 0 for correctly classified samples,
                                                # so w*np.heaviside(-ytr*pred,1) gives the weights of the misclassified samples, 
                                                # and sum(w*np.heaviside(-ytr*pred,1)) gives the total error of the model.
        alpha = 0.5*np.log((1-err)/err) # Alpha is the weight of the model in the final prediction
        
        models[i].alpha = alpha # Store the alpha value in the model
        #alphas[i] = alpha # Store the alpha value in the array of alphas
        
        v = np.zeros((N)) # Update the weights of the samples for the next iteration
        for j in range(N):
            v[j] = w[j]*np.exp(-alpha*ytr[j]*pred[j]) # Update the weight of the sample j
         
        Sm = np.sum(v) # Normalization factor to ensure that the weights sum to 1
        w = v/Sm # Normalize the weights for the next iteration
    return models


def testAdaboost(Xte,yte,models): # For testing, we only need the features and the models
    '''
    The function tests an Adaboost model using decision trees as weak learners.
    Xte: testing features (d x N)
    yte: testing labels (N)
    models: array of trained models
    return: array of predictions
    '''
    nMod = models.shape[0]
    nsamp = Xte.shape[0]
    
    predTot = np.zeros((nMod,nsamp))# Initialize an array to store the weighted predictions of each model
    for m in range(nMod):
        label = models[m].predict(Xte) # Predictions of the model m on the testing data
        predTot[m,:] = models[m].alpha*label # Weighted prediction of the model m
    return np.sign(np.sum(predTot,axis=0)) # Final prediction is the sign of the sum of the weighted predictions of all models

In [7]:


#==========================Main Script==================================
nMod = 1 # Number of models to train
models = trainAdaboost(Xtr.T,ytr,nMod) # Train the Adaboost model

pred = testAdaboost(Xtr.T,ytr,models) # Test the Adaboost model on the training data

#Identify class indexes after prediction
ixW1p=np.where(pred==1)[0]
ixW2p=np.where(pred==-1)[0]

#Calculate accuracy
hits=np.sum(pred==ytr)
AC=hits/Xtr.shape[1]

print("Accuracy="+str(AC))

fig=go.Figure()
fig.add_trace(go.Scatter(x=Xtr[0,ixW1],y=Xtr[1,ixW1],name='w1'))
fig.add_trace(go.Scatter(x=Xtr[0,ixW2],y=Xtr[1,ixW2],name='w2'))


fig.add_trace(go.Scatter(x=Xtr[0,ixW1p],y=Xtr[1,ixW1p],name='Pred as w1',
                         marker_size=10,marker_symbol="circle-open",marker=dict(color="blue",
                                                                                line=dict(width=3,color='blue'))))
fig.add_trace(go.Scatter(x=Xtr[0,ixW2p],y=Xtr[1,ixW2p],name='Pred as w2',
                         marker_size=10,marker_symbol="circle-open",marker=dict(color="red",
                                                                                line=dict(width=3,color='red'))))
fig.update_traces(mode='markers')
fig.update_xaxes(title_text="f1")
fig.update_yaxes(title_text="f2")
fig.update_layout(autosize=False,width=900,height=800)
fig.show()

Accuracy=0.8297161936560935
